In [1]:
import pandas as pd 
import glob

price_file = glob.glob('**/Day-ahead*.csv', recursive=True)[0]
df_price = pd.read_csv(price_file, sep=';', encoding = 'utf-8')

print(df_price.info())
df_price.head()

<class 'pandas.DataFrame'>
RangeIndex: 29184 entries, 0 to 29183
Data columns (total 19 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Start date                                       29184 non-null  str    
 1   End date                                         29184 non-null  str    
 2   Germany/Luxembourg [€/MWh] Original resolutions  29184 non-null  float64
 3   ∅ DE/LU neighbours [€/MWh] Original resolutions  29184 non-null  float64
 4   Belgium [€/MWh] Original resolutions             29184 non-null  str    
 5   Denmark 1 [€/MWh] Original resolutions           29184 non-null  float64
 6   Denmark 2 [€/MWh] Original resolutions           29184 non-null  float64
 7   France [€/MWh] Original resolutions              29184 non-null  float64
 8   Netherlands [€/MWh] Original resolutions         29184 non-null  float64
 9   Norway 2 [€/MWh] Original resolutions  

,Start date,End date,Germany/Luxembourg [€/MWh] Original resolutions,∅ DE/LU neighbours [€/MWh] Original resolutions,Belgium [€/MWh] Original resolutions,Denmark 1 [€/MWh] Original resolutions,Denmark 2 [€/MWh] Original resolutions,France [€/MWh] Original resolutions,Netherlands [€/MWh] Original resolutions,Norway 2 [€/MWh] Original resolutions,Austria [€/MWh] Original resolutions,Poland [€/MWh] Original resolutions,Sweden 4 [€/MWh] Original resolutions,Switzerland [€/MWh] Original resolutions,Czech Republic [€/MWh] Original resolutions,DE/AT/LU [€/MWh] Original resolutions,Northern Italy [€/MWh] Original resolutions,Slovenia [€/MWh] Original resolutions,Hungary [€/MWh] Original resolutions
0,"Jan 1, 2026 12:00 AM","Jan 1, 2026 12:15 AM",63.64,84.75,106.30,63.64,63.63,95.95,72.91,65.60,95.95,109.53,41.58,107.41,109.80,-,111.42,100.51,104.33
1,"Jan 1, 2026 12:15 AM","Jan 1, 2026 12:30 AM",61.48,79.57,87.34,61.48,61.47,82.47,69.16,64.61,89.01,107.86,40.87,107.41,103.58,-,110.58,94.03,98.99
2,"Jan 1, 2026 12:30 AM","Jan 1, 2026 12:45 AM",58.00,74.12,62.92,58.00,57.99,64.96,59.80,63.31,85.61,101.61,40.20,107.41,113.50,-,110.58,92.01,98.83
3,"Jan 1, 2026 12:45 AM","Jan 1, 2026 1:00 AM",49.99,68.45,54.56,49.99,49.98,57.38,51.90,61.22,81.60,80.43,39.23,107.41,119.30,-,107.42,87.38,93.06
4,"Jan 1, 2026 1:00 AM","Jan 1, 2026 1:15 AM",60.01,78.23,78.36,60.01,60.00,78.11,75.66,62.10,89.14,107.86,38.30,106.00,105.00,-,106.01,94.17,99.44


In [2]:
df_price = df_price.iloc[:,[0,1,2]]
df_price.columns = ['start_date', 'end_date', 'spot_price_eur_mwh']
df_price.head()



,start_date,end_date,spot_price_eur_mwh
0,"Jan 1, 2026 12:00 AM","Jan 1, 2026 12:15 AM",63.64
1,"Jan 1, 2026 12:15 AM","Jan 1, 2026 12:30 AM",61.48
2,"Jan 1, 2026 12:30 AM","Jan 1, 2026 12:45 AM",58.00
3,"Jan 1, 2026 12:45 AM","Jan 1, 2026 1:00 AM",49.99
4,"Jan 1, 2026 1:00 AM","Jan 1, 2026 1:15 AM",60.01


In [3]:
df_price['start_date'] = pd.to_datetime(df_price['start_date'], format='mixed')
df_price['end_date'] = pd.to_datetime(df_price['end_date'], format='mixed')

df_price['spot_price_eur_mwh'] = df_price['spot_price_eur_mwh'].astype(str).str.replace(',', '')
df_price['spot_price_eur_mwh'] = pd.to_numeric(df_price['spot_price_eur_mwh'], errors='coerce')

print(df_price.dtypes)
print("'\Missing values:\n", df_price.isnull().sum())

start_date            datetime64[us]
end_date              datetime64[us]
spot_price_eur_mwh           float64
dtype: object
'\Missing values:
 start_date            0
end_date              0
spot_price_eur_mwh    0
dtype: int64


In [4]:
df_price.head()

,start_date,end_date,spot_price_eur_mwh
0,2026-01-01 00:00:00,2026-01-01 00:15:00,63.64
1,2026-01-01 00:15:00,2026-01-01 00:30:00,61.48
2,2026-01-01 00:30:00,2026-01-01 00:45:00,58.00
3,2026-01-01 00:45:00,2026-01-01 01:00:00,49.99
4,2026-01-01 01:00:00,2026-01-01 01:15:00,60.01


In [5]:
import numpy as np

In [6]:
hourly_prices = (
    df_price.groupby(df_price['start_date'].dt.floor('h'))
    .agg(
        spot_price_eur_mwh=('spot_price_eur_mwh', 'mean'),
        price_volatility_15m=('spot_price_eur_mwh', 'std')
    )
    .round(2)
    .reset_index()
)


conditions = [hourly_prices ['spot_price_eur_mwh'] < 0, hourly_prices ['spot_price_eur_mwh'] > 150]
choices = ['Negative Price', 'Price Spike']
hourly_prices['market_regime'] = np.select(conditions, choices, default='Normal Market')

print("total hourly rows:", len(hourly_prices))
print("\n", hourly_prices['market_regime'].value_counts())
hourly_prices.head()

total hourly rows: 7295

 market_regime
Normal Market     4979
Price Spike       1864
Negative Price     452
Name: count, dtype: int64


,start_date,spot_price_eur_mwh,price_volatility_15m,market_regime
0,2026-01-01 00:00:00,58.28,5.99,Normal Market
1,2026-01-01 01:00:00,42.40,15.55,Normal Market
2,2026-01-01 02:00:00,19.57,12.12,Normal Market
3,2026-01-01 03:00:00,6.28,2.80,Normal Market
4,2026-01-01 04:00:00,3.09,2.04,Normal Market


In [7]:
df_gen = pd.read_csv('clean_german_energy.csv')
df_cons = pd.read_csv('clean_energy_consumption.csv')

print("Generation Columns:", df_gen.columns.tolist())
print("\nEnergy Columns:", df_cons.columns.tolist())
print("\nSample Generation Data:", df_gen['start_date'].iloc[0])

Generation Columns: ['start_date', 'end_date', 'biomass_mwh', 'hydropower_mwh', 'wind_offshore_mwh', 'wind_onshore_mwh', 'photovoltaics_mwh', 'other_renewable_mwh', 'nuclear_mwh', 'lignite_mwh', 'hard_coal_mwh', 'fossil_gas_mwh', 'hydro_pumped_storage_mwh', 'other_conventional_mwh']

Energy Columns: ['start_date', 'end_date', 'grid_load_mwh', 'grid_load_incl_hydro_pumped_storage_mwh', 'hydro_pumped_storage_mwh', 'residual_load_mwh']

Sample Generation Data: 2026-01-01 00:00:00


In [8]:
import sqlite3

In [9]:
hourly_prices['start_date'] = hourly_prices['start_date'].astype(str)
df_gen['start_date'] = df_gen['start_date'].astype(str)
df_cons['start_date'] = df_cons['start_date'].astype(str)

conn = sqlite3.connect(':memory:')
hourly_prices.to_sql('prices', conn, index=False, if_exists='replace')
df_gen.to_sql('generation', conn, index=False, if_exists='replace')
df_cons.to_sql('consumption', conn, index=False, if_exists='replace')


join_query = """
SELECT
    p.start_date,
    p.spot_price_eur_mwh,
    p.price_volatility_15m,
    p.market_regime,
    g.wind_onshore_mwh,
    g.wind_offshore_mwh,
    g.photovoltaics_mwh,
    g.fossil_gas_mwh,
    g.lignite_mwh,
    c.grid_load_mwh,
    c.residual_load_mwh
FROM prices p
INNER JOIN generation g ON p.start_date = g.start_date
INNER JOIN consumption c ON p.start_date = c.start_date
ORDER BY p.start_date ASC
"""


master_df = pd.read_sql(join_query, conn)
print("Combined master rows:", len(master_df))
master_df.head()

Combined master rows: 6551


,start_date,spot_price_eur_mwh,price_volatility_15m,market_regime,wind_onshore_mwh,wind_offshore_mwh,photovoltaics_mwh,fossil_gas_mwh,lignite_mwh,grid_load_mwh,residual_load_mwh
0,2026-01-01 00:00:00,58.28,5.99,Normal Market,33124.41,3161.90,15.76,6347.69,3947.69,46122.88,9820.81
1,2026-01-01 01:00:00,42.40,15.55,Normal Market,34691.29,4043.73,15.14,5592.30,3081.94,45598.23,6848.06
2,2026-01-01 02:00:00,19.57,12.12,Normal Market,35449.62,4889.69,15.06,4855.01,2815.75,44673.92,4319.56
3,2026-01-01 03:00:00,6.28,2.80,Normal Market,36594.04,5394.41,16.13,4498.88,2344.65,43796.41,1791.82
4,2026-01-01 04:00:00,3.09,2.04,Normal Market,37100.96,5326.08,12.63,4453.17,2329.38,43359.56,919.90


In [10]:
master_df['total_renewables_mwh'] = (
    master_df['wind_onshore_mwh'] + master_df['wind_offshore_mwh'] + master_df['photovoltaics_mwh']
)
master_df['renewable_share_pct'] = (
    (master_df['total_renewables_mwh'] / master_df['grid_load_mwh']) * 100
    ).round(2)

baseload_price = master_df['spot_price_eur_mwh'].mean()

solar_price = (master_df['photovoltaics_mwh'] * master_df['spot_price_eur_mwh']).sum() / master_df['photovoltaics_mwh'].sum()
onshore_wind_price = (master_df['wind_onshore_mwh'] * master_df['spot_price_eur_mwh']).sum() / master_df['wind_onshore_mwh'].sum()
offshore_wind_price = (master_df['wind_offshore_mwh'] * master_df['spot_price_eur_mwh']).sum() / master_df['wind_offshore_mwh'].sum()

print(f"Baseload Avg Spot Price: €{baseload_price:.2f} / MWh")
print(f"Solar Capture Price: €{solar_price:.2f} / MWh ({solar_price / baseload_price:.1%} of baseload)")
print(f"Onshore Wind Capture Price: €{onshore_wind_price:.2f} / MWh ({onshore_wind_price / baseload_price:.1%} of baseload)")
print(f"Offshore Wind Capture Price: €{offshore_wind_price:.2f} / MWh ({offshore_wind_price / baseload_price:.1%} of baseload)")
    

Baseload Avg Spot Price: €107.72 / MWh
Solar Capture Price: €56.77 / MWh (52.7% of baseload)
Onshore Wind Capture Price: €95.48 / MWh (88.6% of baseload)
Offshore Wind Capture Price: €101.76 / MWh (94.5% of baseload)


In [11]:
master_df.to_sql('master_market', conn, index=False, if_exists='replace')

regime_summary_query = """
SELECT
    market_regime,
    COUNT(*) AS total_hours,
    ROUND(AVG(spot_price_eur_mwh), 2) AS avg_spot_price,
    ROUND(AVG(renewable_share_pct), 1) AS avg_renewable_share_pct,
    ROUND(AVG(residual_load_mwh), 0) AS avg_residual_load_mwh
FROM master_market
GROUP BY market_regime
ORDER BY avg_spot_price DESC;
"""

regime_summary = pd.read_sql(regime_summary_query, conn)
print(regime_summary) 

    market_regime  total_hours  avg_spot_price  avg_renewable_share_pct  \
0     Price Spike         1375          187.69                     26.0   
1   Normal Market         4724           96.39                     52.4   
2  Negative Price          452          -17.19                    108.5   

   avg_residual_load_mwh  
0                39022.0  
1                25158.0  
2                -3789.0  


In [12]:
master_df.to_csv('master_market_data.csv', index=False)
print("\nExported master_market_data.csv successfully!")


Exported master_market_data.csv successfully!
